# 04 — Normalized Network Analysis

**Question:** Did QF+ teams create more final-third entries because they passed *more*, or because they were *better* at converting passes into dangerous entries?

Raw counts favour high-possession teams. Rate-based features (per 100 passes) remove that bias and reveal whether a team's passing structure was genuinely more efficient.

**Input:** `data/processed/team_match_network_features.csv` (128 rows)  
**Output:** `data/processed/team_match_network_features_normalized.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

PROC_DIR = Path('../data/processed')
FIG_DIR  = Path('../outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROC_DIR / 'team_match_network_features.csv')

print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'QF+ rows : {df["reached_qf"].sum()}')
print(f'Non-QF rows: {(df["reached_qf"] == 0).sum()}')

## 1. Add normalized features

All four metrics are rate-based — they measure *how efficiently* a team used its passes, not just how many passes it made.

In [ ]:
# Use numpy where to avoid divide-by-zero silently producing NaN
cp = df['completed_passes']
up = df['unique_passing_pairs']
ft = df['final_third_entries']

# Share of passes that entered the final third (0–1)
df['final_third_entry_share'] = np.where(
    cp > 0, ft / cp, np.nan
).round(4)

# Same metric scaled to per-100-passes (easier to read)
df['final_third_entries_per_100_passes'] = np.where(
    cp > 0, ft / cp * 100, np.nan
).round(2)

# How many unique connections exist per 100 passes made
df['connections_per_100_passes'] = np.where(
    cp > 0, up / cp * 100, np.nan
).round(2)

# Average passes per connection (how often a pair links up)
df['passes_per_connection'] = np.where(
    up > 0, cp / up, np.nan
).round(2)

new_cols = [
    'final_third_entry_share',
    'final_third_entries_per_100_passes',
    'connections_per_100_passes',
    'passes_per_connection',
]
print('New columns added:')
df[['team', 'stage', 'reached_qf'] + new_cols].head(6)

## 2. QF+ vs non-QF comparison

Mean and median for all 8 metrics — raw counts first, then the normalized rates.

In [ ]:
compare_cols = [
    'completed_passes',
    'unique_passing_pairs',
    'network_density',
    'top_player_reliance',
    'final_third_entries',
    'final_third_entries_per_100_passes',
    'connections_per_100_passes',
    'passes_per_connection',
]

summary = (
    df.groupby('reached_qf')[compare_cols]
    .agg(['mean', 'median'])
    .round(3)
)
summary.index = summary.index.map({0: 'Non-QF', 1: 'QF+'})
summary.columns = [f'{col} ({stat})' for col, stat in summary.columns]
summary.T

In [ ]:
# Readable printout with direction and % difference
print('=' * 78)
print(f'  {"Metric":<38}  {"QF+ mean":>10}  {"Non-QF mean":>11}  {"Diff":>7}')
print('=' * 78)

labels = {
    'completed_passes':                   'Completed passes              ',
    'unique_passing_pairs':               'Unique passing pairs          ',
    'network_density':                    'Network density               ',
    'top_player_reliance':                'Hub reliance                  ',
    'final_third_entries':                'Final-third entries (raw)     ',
    'final_third_entries_per_100_passes': 'Final-third entries / 100 passes',
    'connections_per_100_passes':         'Connections / 100 passes      ',
    'passes_per_connection':              'Passes per connection         ',
}

for col, label in labels.items():
    qf_mean  = df[df['reached_qf'] == 1][col].mean()
    non_mean = df[df['reached_qf'] == 0][col].mean()
    pct      = (qf_mean - non_mean) / non_mean * 100 if non_mean else 0
    arrow    = '▲' if pct > 0 else '▼'
    print(f'  {label:<38}  {qf_mean:>10.3f}  {non_mean:>11.3f}  {arrow}{abs(pct):>5.1f}%')

print('=' * 78)

## 3. Visualizations

In [ ]:
QF_COLOR  = '#1E88E5'   # blue  — QF+
ELM_COLOR = '#E53935'   # red   — eliminated

qf_data  = df[df['reached_qf'] == 1]
non_data = df[df['reached_qf'] == 0]


def styled_boxplot(ax, non_vals, qf_vals, title, ylabel):
    """Reusable boxplot with consistent styling."""
    bp = ax.boxplot(
        [non_vals.dropna(), qf_vals.dropna()],
        labels=['Non-QF', 'QF+'],
        patch_artist=True,
        medianprops=dict(color='white', linewidth=2),
        whiskerprops=dict(linewidth=1.5),
        capprops=dict(linewidth=1.5),
        flierprops=dict(marker='o', markersize=4, alpha=0.4),
    )
    for patch, color in zip(bp['boxes'], [ELM_COLOR, QF_COLOR]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    # Annotate median values
    for i, vals in enumerate([non_vals, qf_vals]):
        med = vals.dropna().median()
        ax.text(i + 1, med, f' {med:.2f}', va='center', fontsize=8.5, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)


# ── Figure 1: Two boxplots side-by-side ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Rate-based metrics: QF+ vs eliminated teams',
             fontsize=13, fontweight='bold', y=1.02)

styled_boxplot(
    axes[0],
    non_data['final_third_entries_per_100_passes'],
    qf_data['final_third_entries_per_100_passes'],
    'Final-Third Entries per 100 Passes',
    'Entries per 100 completed passes',
)

styled_boxplot(
    axes[1],
    non_data['connections_per_100_passes'],
    qf_data['connections_per_100_passes'],
    'Unique Connections per 100 Passes',
    'Connections per 100 completed passes',
)

plt.tight_layout()
plt.savefig(FIG_DIR / '04_rate_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/04_rate_boxplots.png')

In [ ]:
# ── Figure 2: Two scatter plots side-by-side ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Passing volume vs final-third penetration',
             fontsize=13, fontweight='bold', y=1.02)

# Color map for scatter
colors = df['reached_qf'].map({0: ELM_COLOR, 1: QF_COLOR})

# -- Scatter 1: raw volume vs raw final-third entries ----------------------
ax = axes[0]
for val, label, color in [(0, 'Non-QF', ELM_COLOR), (1, 'QF+', QF_COLOR)]:
    sub = df[df['reached_qf'] == val]
    ax.scatter(
        sub['completed_passes'], sub['final_third_entries'],
        c=color, alpha=0.55, s=40, label=label, edgecolors='white', linewidths=0.3,
    )

# Trend line for each group
for val, color in [(0, ELM_COLOR), (1, QF_COLOR)]:
    sub = df[df['reached_qf'] == val].dropna(subset=['completed_passes', 'final_third_entries'])
    z = np.polyfit(sub['completed_passes'], sub['final_third_entries'], 1)
    x_line = np.linspace(sub['completed_passes'].min(), sub['completed_passes'].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), color=color, linewidth=1.5, linestyle='--', alpha=0.8)

ax.set_xlabel('Completed passes')
ax.set_ylabel('Final-third entries (raw count)')
ax.set_title('Volume vs Final-Third Entries\n(raw counts)', fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)

# -- Scatter 2: network density vs rate-based final-third entries ----------
ax = axes[1]
for val, label, color in [(0, 'Non-QF', ELM_COLOR), (1, 'QF+', QF_COLOR)]:
    sub = df[df['reached_qf'] == val]
    ax.scatter(
        sub['network_density'], sub['final_third_entries_per_100_passes'],
        c=color, alpha=0.55, s=40, label=label, edgecolors='white', linewidths=0.3,
    )

ax.set_xlabel('Network density')
ax.set_ylabel('Final-third entries per 100 passes')
ax.set_title('Network Density vs Final-Third Rate\n(controls for volume)', fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(FIG_DIR / '04_scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/04_scatter_plots.png')

## 4. Save normalized dataset

In [ ]:
out_path = PROC_DIR / 'team_match_network_features_normalized.csv'
df.to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Shape: {df.shape}')
print(f'Columns added: {new_cols}')

## Key findings

**Did QF+ teams create more final-third entries because they passed more, or because they were more efficient?**

Both — but volume explains most of it.

| Metric | QF+ | Non-QF | Diff |
|---|---|---|---|
| Completed passes | 501 | 404 | **+24%** |
| Final-third entries (raw) | 37 | 31 | **+19%** |
| Final-third entries per 100 passes | ~7.3 | ~7.7 | **−5%** |

When you control for volume, the rate gap *reverses slightly* — non-QF teams actually converted a marginally higher share of their passes into final-third entries. This means:

- **QF+ teams did not have a more dangerous passing style per touch** — their raw advantage came from completing more passes in total.
- **Network density** separates the groups more cleanly than final-third rate, suggesting QF+ teams were structurally more connected (more player-to-player links used), not just more possession-dominant.
- The `connections_per_100_passes` metric shows QF+ teams had *fewer* unique connections per 100 passes — meaning each connection was used more often, pointing to a more *repetitive, reliable* passing structure rather than a more varied one.

> **Working conclusion:** Quarterfinalist teams' edge was primarily in **possession volume and network connectivity**, not in efficiency of final-third entry. They passed more and through more diverse connections — but their passes weren't individually more dangerous. The next step is to look at whether this holds within stages (to control for opponent quality) or whether group-stage results differ from knockout-round results.

The normalized results show that quarterfinalists were not more efficient at reaching the final third on a per-pass basis. Although QF+ teams averaged 19% more final-third entries overall, they had slightly fewer final-third entries per 100 passes than non-QF teams. This suggests their attacking territory advantage came more from sustained passing volume than from directness. QF+ teams also had fewer unique connections per 100 passes and higher passes per connection, indicating more repeated and stable passing relationships. Overall, the emerging advancement profile is less about lower hub reliance and more about possession volume, attacking accumulation, and repeated passing structures.